[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/09_SVD_Recommendation.ipynb)

# MATH 232 - Math Models w/ Tech
## SVD & Recommendation Systems
### Instructor: Prof. Mario Bañuelos 

In [1]:
# Import necessary packages
from IPython.display import HTML, Image
import numpy as np
from scipy import linalg as la
from scipy import sparse
import pandas as pd
# import plotting packages
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

## Outline

* [Singular Value Decomposition](#svd)
* [The Netflix Problem](#netflix)
    * [Cosine Similarity](#cosine)
* [Further Reading](#reading)

# Singular Value Decomposition <a id='svd'></a>

### Warm-Up 1.

* What does it mean for a vector $\vec{v}$ to be orthogonal?
* What does it mean for a matrix to be orthonormal?

Let's assume we are dealing with an $m \times n$ matrix (which will most likely be the case) instead of a square matrix. We cannot solve the eigenvalue problem, but we can still factor the matrix $M$ using the **singular value decomposition**, which is a generalization of orthogonal diagonalization. We seek a factorization of the form,

$$
M = U \Sigma V^T
$$

**Q:** Why/how can we do this?

In [2]:
Image(url="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Singular_value_decomposition_visualisation.svg/1280px-Singular_value_decomposition_visualisation.svg.png", width=400, height=400)

### Low-Rank and Compact SVD Representations

**Compact SVD**
Denoted $M = U_1 \Sigma_1 V_1^T$, $U_1$ is $m \times r$ (the first $r$ columns of $U$),  $V_1$ is $n \times r$ (the first $r$ columns of $V$), and $\Sigma_1$ is $r \times r$. In this case, the zero singular values are omitted.

**Low Rank Representation**
If $M$ is a  $m \times n$ matrix of rank $r$, we seek an approximation of $M$ that will require less memory on a computer. 

### Warm-Up 2.


If a real matrix $A$ has rank $r$, why does the matrix $A^T A$ have the same rank?

**Hint:** Frame the problem as an $A \vec{x} = \vec{0}$ equation?

## Image Compression with SVD

By compressing images, we can calculate the so-called *compression ratio*,

$$
\text{Compression Ratio} = \frac{\text{Uncompressed Size}}{\text{Compressed Size}}
$$

<a href="http://timbaumann.info/svd-image-compression-demo/"> SVD Image Compression Demo </a>

In your groups, pick two of the images and compare how many singular values you need to not be able to tell the difference between the reconstruction and the original picture. Is there a reason why one picture uses more, less, the same amount?

# The Netflix Problem (Prize) <a id='netflix'></a>

In October 2006, <a href="https://www.netflixprize.com/index.html"> Netflix began an open competition </a> to predict user ratings for films with a grand prize of 1 million dollars. 

In September 2009, the grand prize was given to BellKor's Pragmatic Chaos team (improving predicting ratings by 10%)

**Discussion Q:** 
* What does success mean in the context of this competition? 
* How would you frame such a model?
* What potential concerns arise?
* Recommend a movie to me: https://tinyurl.com/drmbmovie

In [2]:
data = pd.io.parsers.read_csv('ml-1m/ratings.dat', 
    names=['user_id', 'movie_id', 'rating', 'time'],
    engine='python', delimiter='::')

movie_data = pd.io.parsers.read_csv('ml-1m/movies.dat',
    names=['movie_id', 'title', 'genre'],
    engine='python', delimiter='::', encoding='latin-1') 

# if data is in same folder as notebook
#data = pd.io.parsers.read_csv('ml-1m/ratings.dat', 
#    names=['user_id', 'movie_id', 'rating', 'time'],
#    engine='python', delimiter='::')

#movie_data = pd.io.parsers.read_csv('ml-1m/movies.dat',
#    names=['movie_id', 'title', 'genre'],
#    engine='python', delimiter='::') 

Create the rating matrix with rows as movies and columns as users.

In [3]:
ratings_mat = np.ndarray(
    shape=(np.max(data.movie_id.values), np.max(data.user_id.values)),
    dtype=np.uint8)
ratings_mat[data.movie_id.values-1, data.user_id.values-1] = data.rating.values

In [5]:
ratings_mat.shape

(3952, 6040)

In [4]:
ratings_mat[0:10,0:10]

array([[5, 0, 0, 0, 0, 4, 0, 4, 5, 5],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 5],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 3, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 2, 0, 4, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 4],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=uint8)

## Cosine Similarity <a id='cosine'></a>

Cosine similarity is a measure of similarity between two non-zero vectors of an inner product space. It is defined to equal the cosine of the angle between them. This metric is often used in natural language processing, text mining, and recommendation systems.

For two non-zero vectors $A,B$, we can derive cosine similarity using the dot product (inner product) formula:

$$
A \cdot B = \| A \| \| B \| \cos (\theta).
$$


Then,

$$
 \cos (\theta) = \frac{ A \cdot B}{ \| A \| \| B \|}.
$$

In [6]:
# normalize matrix
# subtract the mean
normalised_mat = ratings_mat - np.asarray([(np.mean(ratings_mat, 1))]).T
A = normalised_mat.T / np.sqrt(ratings_mat.shape[0] - 1)

In [8]:
A.shape

(6040, 3952)

In [19]:
# compute SVD
U, S, V = np.linalg.svd(A)

Define a function to calculate the cosine similarity. Sort by most similar and return the top N results

In [20]:
from sklearn.metrics.pairwise import cosine_similarity
from numpy.linalg import norm

In [24]:
# define cosine simalirity
def top_cosine_similarity(data, movie_id, top_n=10):
    index = movie_id - 1 # Movie id starts from 1 in the dataset
    movie_row = data[index, :]
    magnitude = np.sqrt(np.einsum('ij, ij -> i', data, data))
    similarity = np.dot(movie_row, data.T) / (magnitude[index] * magnitude)
    print(magnitude[index] * magnitude)
    #similarity = np.dot(movie_row, data.T) / (norm(movie_row) * norm(data, axis=1))

    sort_indexes = np.argsort(-similarity)
    return sort_indexes[:top_n]#, magnitude[index], magnitude

Define a function to print top N similar movies.

In [25]:
def print_similar_movies(movie_data, movie_id, top_indexes):
    print('Recommendations for {0}: \n'.format(
    movie_data[movie_data.movie_id == movie_id].title.values[0]))
    for id in top_indexes + 1:
        print(movie_data[movie_data.movie_id == id].title.values[0])

Initialize the value of $k$ principal components, id of the movie as given in the dataset, and number of top elements to be printed.

In [43]:
k = 20
movie_id = 13 # (getting an id from movies.dat)
top_n = 5
sliced = V.T[:, :k] # representative data
indices = top_cosine_similarity(sliced, movie_id, top_n)

[0.01149404 0.00329881 0.00252856 ... 0.00076291 0.00065558 0.00448255]


/var/folders/nh/b9hv_l4x633380g76yqp7n480000gr/T/ipykernel_4710/2288427981.py:6: RuntimeWarning: invalid value encountered in true_divide
  similarity = np.dot(movie_row, data.T) / (magnitude[index] * magnitude)


In [44]:
print_similar_movies(movie_data, movie_id, indices)

Recommendations for Balto (1995): 

Balto (1995)
Thumbelina (1994)
All Dogs Go to Heaven 2 (1996)
Antz (1998)
Tarzan (1999)


## Further Reading <a id="reading"></a>

* <a href="https://analyticsindiamag.com/singular-value-decomposition-svd-application-recommender-system/?"> Recommender System (SVD & Truncated SVD) </a>
* <a href="https://analyticsindiamag.com/singular-value-decomposition-svd-application-recommender-system/">Singular Value Decomposition (SVD) & Its Application In Recommender System
</a>
* <a href="https://www.youtube.com/watch?v=Nx0lRBaXoz4"> Strang Lecture on SVD </a>